In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

In [ ]:
# 1. Load the Dataset
# Assume dataset ka naam 'Mall_Customers.csv' hai
# df = pd.read_csv('Mall_Customers.csv')

import kagglehub
path = kagglehub.dataset_download("shwetabh123/mall-customers")

store_path = path + "/Mall_Customers.csv"
df = pd.read_csv(store_path)
df.head()

# STEP 1: Exploratory Data Analysis (EDA) 

In [ ]:
print("Dataset ki basic info:")
print(df.info())
print("\nDataset ka statistical summary:")
print(df.describe())

In [ ]:
# Gender distribution plot
plt.figure(figsize=(6,4))
sns.countplot(x='Genre', data=df) # Dataset mein column ka naam 'Genre' ya 'Gender' ho sakta hai
plt.title('Gender Distribution')
plt.show()

In [ ]:
# Features ka distribution aur unka aapas mein relation
plt.figure(figsize=(15,5))
sns.pairplot(df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)', 'Genre']], hue='Genre')
plt.show()

In [ ]:
# KDE Plot for Annual Income aur Spending Score
plt.figure(figsize=(10,6))
sns.kdeplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', hue='Genre', fill=True)
plt.title('Annual Income vs Spending Score Distribution by Gender')
plt.show()

In [ ]:
# Data Preprocessing: Clustering ke liye features select karna (Income aur Spending Score)
X = df[['Annual Income (k$)', 'Spending Score (1-100)']].values

# Feature Scaling (Zaroori hai clustering models ke liye)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# STEP 2: K-Means Clustering

In [ ]:
# WCSS (Within-Cluster Sum of Square) find karna Elbow Method ke zariye optimal K ke liye
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

In [ ]:
# Elbow Graph plot karna
plt.figure(figsize=(8,5))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--')
plt.title('Elbow Method For Optimal K')
plt.xlabel('Number of clusters (K)')
plt.ylabel('WCSS')
plt.show()

In [ ]:
# Graph dekh kar pata chalta hai ke K=5 optimal hai
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42)
clusters = kmeans.fit_predict(X_scaled)

In [ ]:

# Clusters ko original dataframe mein add karna
df['Cluster'] = clusters

# STEP 3: Dimensionality Reduction & Visualization (PCA) 

In [ ]:
# Yahan hum 3 variables (Age, Income, Spending) use kar ke PCA apply karte hain taake 2D mein dekh sakein
X_3D = df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].values
X_3D_scaled = scaler.fit_transform(X_3D)

In [ ]:
# PCA apply karna (3 dimensions ko 2 dimensions mein convert karna)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_3D_scaled)

In [ ]:
# PCA Clusters Plot
plt.figure(figsize=(10,6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df['Cluster'], palette='viridis', s=100)
plt.title('Customer Segments using PCA (2D Visualization)')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(title='Cluster')
plt.show()

In [ ]:
# Optional: t-SNE Visualization
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_3D_scaled)

plt.figure(figsize=(10,6))
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=df['Cluster'], palette='Set1', s=100)
plt.title('Customer Segments using t-SNE')
plt.xlabel('t-SNE Feature 1')
plt.ylabel('t-SNE Feature 2')
plt.legend(title='Cluster')
plt.show()